<a href="https://colab.research.google.com/github/sharvani1357/RAG/blob/main/Faiss_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
chunks = [

    "Employees receive 12 casual leaves annually.",

    "Employees receive 15 sick leaves annually.",

    "Employees may work from home twice per week.",

    "Travel expenses are reimbursed within 30 days.",

    "All employees are covered under company medical insurance."

]

print("Total Chunks:", len(chunks))

Total Chunks: 5


In [3]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")
print("Embedded model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedded model loaded


In [4]:
embeddings=model.encode(chunks)
print("Embedding shape:")
print(embeddings.shape)

Embedding shape:
(5, 384)


Create FAISS Index

In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 82.5 MB/s eta 0:00:00


In [5]:
import faiss
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
print("faiss index created")

faiss index created


Add Embedding to faiss

In [6]:
import numpy as np
index.add(np.array(embeddings,dtype=np.float32))
print("vectors stored:",index.ntotal)

vectors stored: 5


User Query

In [8]:
query="How many casual leaves are allowed?"
print(query)

How many casual leaves are allowed?


Convert Query into Embeddings

In [9]:
query_embedding=model.encode([query])

Search Faiss

In [10]:
k=3
distances,indices=index.search(
    np.array(
        query_embedding,
        dtype=np.float32
),k)

In [11]:
print("Search Results:")
print("="*50)
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"Rank {rank+1}:")
    print(f"  Distance: {dist:.4f}")
    print(f"  Chunk Index: {idx}")
    print(f"  Chunk Content: {chunks[idx]}")
    print('-'*50)

Search Results:
Rank 1:
  Distance: 0.6896
  Chunk Index: 0
  Chunk Content: Employees receive 12 casual leaves annually.
--------------------------------------------------
Rank 2:
  Distance: 1.1780
  Chunk Index: 1
  Chunk Content: Employees receive 15 sick leaves annually.
--------------------------------------------------
Rank 3:
  Distance: 1.6475
  Chunk Index: 2
  Chunk Content: Employees may work from home twice per week.
--------------------------------------------------
